# 08 -- Head-to-head between arm champions (plan v5 Step 12d)

Each arm's best-rated checkpoint plays every other one: every pair plays `GAMES_PER_PAIR` games (40) from
shared random openings with colors swapped, and every pair gets the same openings. Reported per pair: score,
W/D/L, the implied Elo difference with a 95% bootstrap CI, and how the draws happened.

These games never enter the reference-set fit: every model's Elo still comes from the frozen opponents alone.
They answer a different question -- how the arms do **against each other** -- and catch what ratings against
the same few opponents can miss (e.g. two models level on Elo, one consistently beating the other).

Every game goes to `results/games.csv` (`kind` = `h2h`, `subject` = `<match>|<a>|<b>`), saved after each pair;
a rerun skips pairs already played. Runs on a GPU pod (the LLM players), from whichever repo clone it is opened
in. New arms join with their best-rated checkpoint as they finish (Steps 14-16).

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
import os
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import yaml
import torch
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod for the LLM players"
print("GPU:", GPU)

## Players and settings
`PLAYERS`: one spec per arm (`eval.agents.load_player`): the row name it has in `results.csv`, its kind
(`sft`, `llm_rl`, `alphazero`), run folder, step or round, and for AlphaZero the search budget (0 = no search).
First round (plan v5 Step 12d, Oct 31; AlphaZero r109 replaces r53 after stage 2, Nov 1). `MATCH` names this
round in `games.csv`; `MATCH` and `GAMES_PER_PAIR` can be set from the shell.

In [ ]:
import json

MATCH = os.environ.get("MATCH", "round1")
GAMES_PER_PAIR = int(os.environ.get("GAMES_PER_PAIR", "40"))
PLAYERS = [
    {"name": "structured_masked_nall_e10_s0@15000", "kind": "sft",
     "run": "structured_masked_nall_e10_s0", "step": 15000},                                # SFT, 706
    {"name": "rl_structured_masked_sft25000_s0_curric120_lr5x@r15", "kind": "llm_rl",
     "run": "structured_masked_sft25000_s0_curric120_lr5x", "step": 15},                   # RL run 4, 697
    {"name": "az_b6c64_s0@r109", "kind": "alphazero",
     "run": "az_b6c64_s0", "step": 109, "simulations": 50},                                  # AlphaZero, 798
]
print("match:", MATCH, "|", GAMES_PER_PAIR, "games per pair")
for p in PLAYERS:
    print(" ", json.dumps(p))

## Play
Loads every player, then plays the pairs not yet in `games.csv`, saving each pair's games as soon as it ends.

In [ ]:
from eval.agents import load_player
from eval.anchors import ResultsTable
from eval.match import head_to_head

eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))

def subject(a, b):
    a, b = sorted((a, b))
    return f"{MATCH}|{a}|{b}"

saved = {}
for g in results.games(kind="h2h"):
    if g["subject"].startswith(MATCH + "|"):
        saved[g["subject"]] = saved.get(g["subject"], 0) + 1
names = [p["name"] for p in PLAYERS]
done = {frozenset((a, b)) for i, a in enumerate(names) for b in names[i + 1:]
        if saved.get(subject(a, b), 0) >= GAMES_PER_PAIR}
print("pairs already played:", [sorted(p) for p in done])

players = {p["name"]: load_player(p, ROOT) for p in PLAYERS}

def save_pair(a, b, games):
    results.log_games(games, "h2h", subject(a, b))
    results.save()

head_to_head(players, GAMES_PER_PAIR, skip=done, on_pair=save_pair)
del players
torch.cuda.empty_cache()

## Table
From `games.csv` (this match's games only). `elo_diff` = a minus b, from a's score: -400 log10(1/score - 1);
`at_edge` = a score of 0 or 1, where the gap is only a bound. With 40 games the CI is roughly +/-100-150 Elo.

In [ ]:
import pandas as pd
from eval.match import head_to_head_table

games = [g for g in ResultsTable(results.path).games(kind="h2h") if g["subject"].startswith(MATCH + "|")]
table = pd.DataFrame(head_to_head_table(games))
pd.set_option("display.max_colwidth", 80)
display(table)